In [76]:
import math
from rich import print
import pandas as pd
import numpy as np


In [77]:
# Set random seed for reproducibility
np.random.seed(42)

n_samples = 100

outlooks = ["Sunny", "Overcast", "Rain"]
temperatures = ["Hot", "Mild", "Cool"]
humidities = ["High", "Normal"]
winds = ["Weak", "Strong"]

# Generate feature distributions
outlook_samples = np.random.choice(outlooks, size=n_samples, p=[0.35, 0.30, 0.35])
temp_samples = np.random.choice(temperatures, size=n_samples, p=[0.30, 0.45, 0.25])
humidity_samples = np.random.choice(humidities, size=n_samples, p=[0.50, 0.50])
wind_samples = np.random.choice(winds, size=n_samples, p=[0.55, 0.45])

# Generate PlayTennis target based on classic decision rules with slight noise
play_tennis = []
for o, t, h, w in zip(
    outlook_samples, temp_samples, humidity_samples, wind_samples
):
    if o == "Overcast":
        prob_yes = 0.95  # Overcast is almost always Yes
    elif o == "Sunny":
        prob_yes = (
            0.15 if h == "High" else 0.90
        )  # High humidity makes sunny bad for tennis
    elif o == "Rain":
        prob_yes = 0.10 if w == "Strong" else 0.85  # Strong wind makes rain bad

    play_tennis.append("Yes" if np.random.rand() < prob_yes else "No")

# Create DataFrame
df = pd.DataFrame(
    {
        "Outlook": outlook_samples,
        "Temperature": temp_samples,
        "Humidity": humidity_samples,
        "Wind": wind_samples,
        "PlayTennis": play_tennis,
    }
)

# Preview dataset summary and first few rows
print(f"Dataset shape: {df.shape}")
print("\nPlayTennis Class Distribution:")
print(df["PlayTennis"].value_counts())
print("\nFirst 10 samples:")
print(df.head(10))


Dataset shape: (100, 5)

PlayTennis Class Distribution:

PlayTennis
Yes    67
No     33
Name: count, dtype: int64

First 10 samples:

Outlook Temperature Humidity    Wind PlayTennis
0  Overcast         Hot   Normal    Weak        Yes
1      Rain        Mild     High    Weak         No
2      Rain        Mild     High    Weak        Yes
3  Overcast        Mild   Normal  Strong        Yes
4     Sunny        Cool   Normal  Strong        Yes
5     Sunny         Hot     High  Strong         No
6     Sunny        Mild     High    Weak         No
7      Rain        Cool   Normal    Weak        Yes
8  Overcast         Hot     High  Strong        Yes
9      Rain         Hot     High    Weak        Yes

In [82]:
class DecisionTree():
  def __init__(self, data):
    self.data = data
    self.entropy = self.calc_entropy(self.data)
    self.H = sum(self.entropy["PlayTennis"].values())
    self.entropy.pop("PlayTennis")
    self.gain = self.calc_gain(self.data)
    self.name = max(self.gain, key=self.gain.get)
    self.branches = self.make_branches()


  def calc_entropy(self, data):
    entropy = {}
    for i in data.columns:
      clases = {}
      sum = 0
      for j in data[i].unique():
        n = (df[i] == j).sum()
        p_yes = ((df[i] == j) & (df["PlayTennis"] == "Yes")).sum()/n
        p_no = 1-p_yes
        if p_yes==1:
          clases[j] = -p_yes*math.log2(p_yes)
        elif p_yes==0:
          clases[j] = -p_no*math.log2(p_no)
        else:
          clases[j] = -p_yes*math.log2(p_yes) -p_no*math.log2(p_no)
      entropy[i] = clases
      n_yes = (df["PlayTennis"] == "Yes").sum()/len(df)
      n_no = 1-n_yes
      py = -n_yes*math.log2(n_yes)
      pn = -n_no*math.log2(n_no)
      entropy["PlayTennis"] = {"Yes": py, "No": pn}
    return entropy

  def calc_gain(self, data):
    gain = {}
    for i in data.columns:
      sum = 0
      if i == "PlayTennis":
        continue
      for j in data[i].unique():
        sum += self.entropy[i][j] * (df[i] == j).sum()/len(data)
      gain[i] = self.H-sum
    return gain

  def get_next(self, name="None"):
    names = {key:value for key,value in self.gain.items() if key!=name}
    return min(self.data, key=data.get)

  def make_branches(self):
    branches = {}

    if len(self.entropy)==1:
      for name in self.entropy[self.name].keys():
        if self.entropy[self.name][name] == 1:
          branches[name] = "Yes"
        if self.entropy[self.name][name] == 0:
          branches[name] = "No"
        else:
          n = (self.data[self.name] == name).sum()
          p = ((self.data[self.name] == name) & (self.data["PlayTennis"] == "Yes")).sum()/n
          branches[name] = "Yes" if p>=0.5 else "No"

    else:
      for name in self.entropy[self.name].keys():
        if self.entropy[self.name][name] == 1:
          branches[name] = "Yes"
        if self.entropy[self.name][name] == 0:
          branches[name] = "No"
        else:
          df = self.data[self.data[self.name] == name]
          branches[name] = DecisionTree(df.drop(columns=[self.name]))

    return branches

  def pred(self, x):
    attribute_value = x[self.name]

    if attribute_value not in self.branches:
        return getattr(self, 'default_prediction', None)

    child = self.branches[attribute_value]
    if isinstance(child, DecisionTree):
        return child.pred(x)
    else:
        return child

  def predict(self, data):
    y = []
    if hasattr(data, 'to_dict'):
        data = data.to_dict(orient='records')
    for x in data:
      y.append(self.pred(x))
    return y



In [83]:
tree = DecisionTree(df)

In [84]:
print(tree.gain)
print(tree.entropy)

{
    'Outlook': np.float64(0.09435034527917385),
    'Temperature': np.float64(0.036585698175244596),
    'Humidity': np.float64(0.050375709823029746),
    'Wind': np.float64(0.050111640557868986)
}

{
    'Outlook': {
        'Overcast': np.float64(0.3809465857053901),
        'Rain': np.float64(0.9886994082884974),
        'Sunny': np.float64(0.9788698505067783)
    },
    'Temperature': {
        'Hot': np.float64(0.9971803988942642),
        'Mild': np.float64(0.7592757847892834),
        'Cool': np.float64(0.9182958340544896)
    },
    'Humidity': {'Normal': np.float64(0.7676515870125797), 'High': np.float64(0.998363672593813)},
    'Wind': {'Weak': np.float64(0.7568336324184035), 'Strong': np.float64(0.9967916319816366)}
}

In [85]:
test = {
    'Outlook': ['Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Rain'],
    'Temperature': [ 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild'],
    'Humidity': ['High', 'Normal', 'High','High', 'Normal', 'Normal'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong'],
}

tree.predict(pd.DataFrame(test))

['No', 'Yes', 'Yes', 'No', 'Yes', 'Yes']